# End-to-End Evaluation Example

This notebook demonstrates the complete `ai4c-scribe` workflow using the CLI:

1. **Extract** PR data from a GitHub repository
2. **Create review cases** for LLM training
3. **Set up the runner** to replay fixing an issue

We use `ai4curation/issue-pr-test-repo` - a small, frozen repo designed for testing.

## Prerequisites

Make sure you have:
- `ai4c-scribe` installed (`uv pip install ai4c-scribe`)
- GitHub CLI (`gh`) authenticated (`gh auth login`)

## Setup: GitHub Authentication

When running in Jupyter, subprocess calls may not have access to `gh`'s keyring authentication.
We set `GITHUB_TOKEN` in the environment to ensure auth works.

In [1]:
import os
import subprocess

# Get GitHub token from gh CLI and set it in the environment
result = subprocess.run(["gh", "auth", "token"], capture_output=True, text=True)
if result.returncode == 0 and result.stdout.strip():
    os.environ["GITHUB_TOKEN"] = result.stdout.strip()
    print("✅ GITHUB_TOKEN set from gh auth")
else:
    print("❌ Failed to get token. Run: gh auth login")
    print(f"   Error: {result.stderr}")

✅ GITHUB_TOKEN set from gh auth


In [2]:
%%bash
# Verify gh authentication
gh auth status

github.com
  ✓ Logged in to github.com account cmungall (GITHUB_TOKEN)
  - Active account: true
  - Git operations protocol: ssh
  - Token: gho_************************************
  - Token scopes: 'admin:public_key', 'gist', 'project', 'read:org', 'repo', 'workflow'

  ✓ Logged in to github.com account cmungall (keyring)
  - Active account: false
  - Git operations protocol: ssh
  - Token: gho_************************************
  - Token scopes: 'admin:public_key', 'gist', 'project', 'read:org', 'repo', 'workflow'

  ✓ Logged in to github.com account dragon-ai-agent (keyring)
  - Active account: false
  - Git operations protocol: ssh
  - Token: gho_************************************
  - Token scopes: 'admin:public_key', 'gist', 'read:org', 'repo'


In [3]:
%%bash
# Verify ai4c-scribe installation
ai4c-scribe --help | head -15

                                                                                
 Usage: ai4c-scribe [OPTIONS] COMMAND [ARGS]...                             
                                                                                
 ai4c-scribe: Learns best practice from your github repo                    
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │
│ --show-completion             Show completion for the current shell, to copy │
│                               it or customize the installation.              │
│ --help                        Show this message and exit.                    │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ extract               Extract PRs 

## Step 1: Extract PRs

Extract PR data from the test repository. We'll extract PR #11 which closes issue #10.

In [4]:
%%bash
mkdir -p output

# Extract PRs from the test repo (limit to 5 for speed)
ai4c-scribe extract ai4curation/issue-pr-test-repo \
    -o output/prs.jsonl \
    -l 5

Mining merged PRs from ai4curation/issue-pr-test-repo (limit: 5)...
Successfully mined 5 PRs

✅ Mining complete!
📊 Results saved to: output/prs.jsonl
📈 Total records: 5

Category breakdown:
  merged_no_mods: 3
  merged_with_mods: 2

🔗 One-to-one issue mappings: 5
⏱️  Average time to merge: 0.1 hours


### Inspect the extracted data

Each line in the JSONL file is a complete PR record with commits, reviews, and linked issues.

In [5]:
%%bash
# Count PRs extracted
echo "PRs extracted: $(wc -l < output/prs.jsonl)"

# Show PR numbers and titles
echo ""
echo "PR summary:"
cat output/prs.jsonl | jq -r '[.pr_number, .metadata.title] | @tsv'

PRs extracted:        5

PR summary:
13	Add issue & PR summary to README
11	Add dragon poem
9	Add maintainer contact to README
6	Add status badges to README
5	Add deployment configuration


In [6]:
%%bash
# Look at PR #11 in detail
cat output/prs.jsonl | jq 'select(.pr_number == 11) | {
    pr_number,
    title: .metadata.title,
    category,
    total_commits: .commits.total_commits,
    linked_issues: [.linked_issues.issues[].number]
}'

jq: error (at <stdin>:2): Cannot iterate over null (null)


## Step 2: Create Review Cases

Review cases capture the state at "first revision" - the diff before any reviewer feedback.

In [7]:
%%bash
# Create review cases from extracted PRs
ai4c-scribe create-review-cases output/prs.jsonl \
    -o output/review-cases.jsonl

Creating review cases from output/prs.jsonl...

✅ Review case creation complete!
📊 Results saved to: output/review-cases.jsonl (format: jsonl)
📈 Input records: 5
📝 Review cases created: 2
⏭️  Skipped (no reviews): 3


In [8]:
%%bash
# Inspect review cases
echo "Review cases created: $(wc -l < output/review-cases.jsonl)"
echo ""

# Show structure of a review case
head -1 output/review-cases.jsonl | jq 'keys'

Review cases created:        2

[
  "cumulative_diff_at_first_review",
  "first_revision_action",
  "first_revision_date",
  "first_revision_reviews",
  "issue_comments_before_pr",
  "linked_issue_body",
  "linked_issue_number",
  "linked_issue_title",
  "num_reviews_in_first_revision",
  "parent_commit_sha",
  "pr_body",
  "pr_comments_before_review",
  "pr_created_at",
  "pr_number",
  "pr_title",
  "repository"
]


### Export as Markdown

For human review, export review cases as readable markdown.

In [9]:
%%bash
# Create markdown version
ai4c-scribe create-review-cases output/prs.jsonl \
    -o output/review-cases.md \
    -f markdown

# Show first 50 lines
head -50 output/review-cases.md

Creating review cases from output/prs.jsonl...

✅ Review case creation complete!
📊 Results saved to: output/review-cases.md (format: markdown)
📈 Input records: 5
📝 Review cases created: 2
⏭️  Skipped (no reviews): 3
# Review Case: ai4curation/issue-pr-test-repo PR #11

**Title:** Add dragon poem
**Created:** 2025-12-23 22:49:29 UTC
**First Review:** 2025-12-23 22:50:31 UTC
**Action:** CHANGES_REQUESTED
**Reviews in First Revision:** 1

## PR Description

*No description provided*

## Linked Issue #10

**Title:** create a file poem.md

### Issue Description

create a file poem.md with a 2 line poem about dragons

*No comments before PR creation*

## Parent Commit

**SHA:** `97b4fc884fa1fce1afcf3cc8dd74dd6309839e64`

This is the commit immediately before the PR branch was created.

## Cumulative Diff at First Review

All changes made up to the point of the first review:

```diff
# Commit: cf3ba409 - Add dragon poem
@@ -0,0 +1,4 @@
+# Dragon Poem
+
+Scales of fire, wings of night,
+A drag

## Step 3: Set Up the Runner

The runner prepares a worktree to replay fixing an issue. This lets you:
- Reset to the state *before* the PR
- Run an AI agent to fix the issue
- Compare the agent's solution to the ground truth

First, clone the test repo:

In [10]:
%%bash
# Clone the test repo (if not already cloned)
if [ ! -d output/issue-pr-test-repo ]; then
    git clone https://github.com/ai4curation/issue-pr-test-repo.git output/issue-pr-test-repo
else
    echo "Repo already cloned"
fi

Repo already cloned


### Create a runner configuration

The runner needs a config file with the experiment ID and system prompt.

In [11]:
%%bash
# Create runner config
mkdir -p output/issue-pr-test-repo/.ai4cscribe

cat > output/issue-pr-test-repo/.ai4cscribe/runner.yaml << 'EOF'
experiment_id: eval-001
system_prompt: |
  You are a helpful coding assistant.
  Follow the issue instructions carefully.
  Make minimal, focused changes.
agent_timeout: 300
EOF

cat output/issue-pr-test-repo/.ai4cscribe/runner.yaml

experiment_id: eval-001
system_prompt: |
  You are a helpful coding assistant.
  Follow the issue instructions carefully.
  Make minimal, focused changes.
agent_timeout: 300


### Run fix-issue (dry run)

Use `--dry-run` to see what would happen without actually running the agent.

In [12]:
%%bash
# Fix issue #10 (which is addressed by PR #11)
# Use --dry-run to just set up without running the agent
ai4c-scribe fix-issue ai4curation/issue-pr-test-repo 10 output/issue-pr-test-repo --dry-run

🔧 Fixing issue #10 from ai4curation/issue-pr-test-repo...
📁 Worktree: output/issue-pr-test-repo
🏃 Dry run mode: will set up worktree but not run agent



❌ Error: Command '['git', 'reset', '--hard', '97b4fc884fa1fce1afcf3cc8dd74dd6309839e64']' returned non-zero exit status 128.
❌ Error fixing issue: 


CalledProcessError: Command 'b'# Fix issue #10 (which is addressed by PR #11)\n# Use --dry-run to just set up without running the agent\nai4c-scribe fix-issue ai4curation/issue-pr-test-repo 10 output/issue-pr-test-repo --dry-run\n'' returned non-zero exit status 1.

### Verify the setup

In [ ]:
%%bash
cd output/issue-pr-test-repo

echo "Current branch:"
git branch --show-current

echo ""
echo "Current commit:"
git log -1 --oneline

echo ""
echo "Files in repo:"
ls -la | grep -v '^\.'

## Step 4: Compare with Ground Truth

After running the agent, compare its output with the actual PR #11.

In [ ]:
%%bash
# Show the ground truth - what PR #11 actually changed
echo "Ground truth (PR #11 diff):"
echo "============================"
gh pr diff 11 --repo ai4curation/issue-pr-test-repo

## Cache Management

`ai4c-scribe` caches GitHub API responses to speed up repeated runs.

In [ ]:
%%bash
# View cache stats
ai4c-scribe cache stats

In [ ]:
%%bash
# View cache for specific repo
ai4c-scribe cache stats --repo ai4curation/issue-pr-test-repo

## Summary

This notebook demonstrated the complete `ai4c-scribe` CLI workflow:

| Command | Purpose |
|---------|--------|
| `ai4c-scribe extract` | Mine PRs from a GitHub repo |
| `ai4c-scribe create-review-cases` | Create training cases from PRs |
| `ai4c-scribe fix-issue` | Set up runner to replay a fix |
| `ai4c-scribe cache stats` | View cache statistics |

For Python API usage, see the [Python API Example](python_api_example.ipynb) notebook.

## Cleanup

In [ ]:
%%bash
# Uncomment to clean up
# rm -rf output/
# echo "Cleaned up output directory"